1. Data Cleaning

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *

spark = SparkSession.builder.appName("Data Clean").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')

data = [
	(1, 'Alice', 25, 'NY'),
	(2, 'Bob', None, 'LA'),
	(2, 'Bob', None, 'LA'),
	(3, None, 40, 'NY')
]

schema = StructType([
	StructField('id', IntegerType()),
	StructField('name', StringType()),
	StructField('age', IntegerType()),
	StructField('city', StringType())
])

people = spark.createDataFrame(data, schema=schema).dropDuplicates()

avg_age = people.agg(func.mean('age')).first()[0]
people_fix_age = people.fillna(avg_age, subset=['age'])
people_fix_name = people_fix_age.fillna('Unknown', subset=['name'])

people_final = people_fix_name.filter(people_fix_name.age >  30)

people_final.show()




+---+-------+---+----+
| id|   name|age|city|
+---+-------+---+----+
|  2|    Bob| 32|  LA|
|  3|Unknown| 40|  NY|
+---+-------+---+----+



2. Group By and Aggregation

In [17]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *

spark = SparkSession.builder.appName("Group By Agg").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
	('A', 'Phone', 500),
	('A', 'Laptop', 1000),
	('B', 'Phone', 700)
]

schema = StructType([
	StructField('customer', StringType()),
	StructField('product', StringType()),
	StructField('amount', IntegerType()),
])

sales = spark.createDataFrame(data=data, schema=schema)

totals_per_customer = sales.groupBy('customer').agg(func.sum(func.col('amount')).alias('totals'))

totals_per_customer.show()

avg_purchase = sales.agg(func.avg(func.col('amount')).alias('Average Purchase'))

avg_purchase.show()

max_spenders = sales.groupBy('customer').agg(func.sum(func.col('amount')).alias('total')).sort('total', ascending=False)

max_spenders.show(1)


+--------+------+
|customer|totals|
+--------+------+
|       A|  1500|
|       B|   700|
+--------+------+

+-----------------+
| Average Purchase|
+-----------------+
|733.3333333333334|
+-----------------+

+--------+-----+
|customer|total|
+--------+-----+
|       A| 1500|
+--------+-----+
only showing top 1 row


3. Top N per Group

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Top N").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
	('IT', 'John', 100),
	('IT', 'Mary', 200),
	('IT', 'Alex', 150)
]

schema = StructType([
	StructField('department', StringType()),
	StructField('employee', StringType()),
	StructField('salary', IntegerType()),
])

emplys = spark.createDataFrame(data=data, schema=schema)

windowSpec = Window.partitionBy("department").orderBy(func.col("salary").desc())

result_df = emplys.withColumn("rn", func.row_number().over(windowSpec)) \
              .filter(func.col("rn") == 1) \
              .drop("rn")

result_df.show()



+----------+--------+------+
|department|employee|salary|
+----------+--------+------+
|        IT|    Mary|   200|
+----------+--------+------+



4. Joins

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Joins").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data1 = [
	(1, 'John'),
	(2, 'Mary')
]

schema1 = StructType([
	StructField('id', IntegerType()),
	StructField('name', StringType())
])

data2 = [
	(1, 100.0),
	(1, 50.0),
	(3, 25.0)
]

schema2 = StructType([
	StructField('id', IntegerType()),
	StructField('amount', FloatType())
])

customers = spark.createDataFrame(data=data1, schema=schema1)
orders = spark.createDataFrame(data=data2, schema=schema2)

#Inner Join
customers.join(orders, on='id').show()

#Left Join
customers.join(orders, on='id', how='left').show()

#Left Anti Join
customers.join(orders, on='id', how='left_anti').show()

#Totals
customers.join(orders.groupBy('id').agg(func.sum(func.col('amount')).alias('total')), on='id', how='left').show()



+---+----+------+
| id|name|amount|
+---+----+------+
|  1|John| 100.0|
|  1|John|  50.0|
+---+----+------+

+---+----+------+
| id|name|amount|
+---+----+------+
|  1|John|  50.0|
|  1|John| 100.0|
|  2|Mary|  NULL|
+---+----+------+

+---+----+
| id|name|
+---+----+
|  2|Mary|
+---+----+

+---+----+-----+
| id|name|total|
+---+----+-----+
|  1|John|150.0|
|  2|Mary| NULL|
+---+----+-----+



5. Find Duplicate Records

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Duplicate").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
	('a@test.com',),
	('b@test.com',),
	('a@test.com',),
	('c@test.com',),
]

schema = StructType([
	StructField('email', StringType())
])

emails = spark.createDataFrame(data=data, schema=schema)

emails.groupBy('email').count().show()

emails.dropDuplicates().show()

+----------+-----+
|     email|count|
+----------+-----+
|a@test.com|    2|
|b@test.com|    1|
|c@test.com|    1|
+----------+-----+

+----------+
|     email|
+----------+
|a@test.com|
|b@test.com|
|c@test.com|
+----------+



6. Window Function Challenge

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Window").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
	("A", "Jan1", 10),
	("A", "Jan2", 20),
	("A", "Jan3", 15)
]

columns = ['user', 'date', 'amount']

transactions = spark.createDataFrame(data=data, schema=columns)

#transactions.show()

total_counter = Window.partitionBy("user").orderBy(func.col("date")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

transactions_with_total = transactions.withColumn('running_total', func.sum('amount').over(windowSpec))

transactions_with_total.show()

tracker = Window.partitionBy("user").orderBy(func.col("date"))

transactions_with_prev = transactions.withColumn('previous', func.lag('amount', 1).over(tracker))

transactions_with_prev.show()

transactions_with_prev_diff = transactions_with_prev.withColumn('difference', func.col('amount') - func.col('previous'))

transactions_with_prev_diff.show()



+----+----+------+-------------+
|user|date|amount|running_total|
+----+----+------+-------------+
|   A|Jan1|    10|           10|
|   A|Jan2|    20|           30|
|   A|Jan3|    15|           45|
+----+----+------+-------------+

+----+----+------+--------+
|user|date|amount|previous|
+----+----+------+--------+
|   A|Jan1|    10|    NULL|
|   A|Jan2|    20|      10|
|   A|Jan3|    15|      20|
+----+----+------+--------+

+----+----+------+--------+----------+
|user|date|amount|previous|difference|
+----+----+------+--------+----------+
|   A|Jan1|    10|    NULL|      NULL|
|   A|Jan2|    20|      10|        10|
|   A|Jan3|    15|      20|        -5|
+----+----+------+--------+----------+



7. Sessionization

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Session").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
	(1, "2026-03-29 10:00:00"),
    (1, "2026-03-29 10:15:00"),  # 15 min gap (same session)
    (1, "2026-03-29 10:50:00"),  # 35 min gap (NEW session)
    (1, "2026-03-29 11:00:00"),  # 10 min gap (same session)
    (2, "2026-03-29 10:00:00"),  # New user (NEW session)
    (2, "2026-03-29 10:40:00"),
]

columns = ['user', 'timestamp']

users = spark.createDataFrame(data=data, schema=columns)
users = users.withColumn("timestamp", func.col("timestamp").cast("timestamp"))
window_spec = Window.partitionBy("user").orderBy("timestamp")
users = users.withColumn("prev_timestamp", func.lag("timestamp").over(window_spec))
users = users.withColumn(
    "gap_seconds", 
    func.col("timestamp").cast("long") - func.col("prev_timestamp").cast("long")
)
users = users.withColumn(
    "is_new_session",
    func.when((func.col("gap_seconds").isNull()) | (func.col("gap_seconds") > 1800), 1).otherwise(0)
)
cumulative_window = Window.partitionBy("user").orderBy("timestamp").rowsBetween(Window.unboundedPreceding, Window.currentRow)

users = users.withColumn("user_session_idx", func.sum("is_new_session").over(cumulative_window))
users = users.withColumn("session_id", func.concat_ws("_", func.col("user"), func.col("user_session_idx"))).drop('user_session_idx')
users.show()

+----+-------------------+-------------------+-----------+--------------+----------+
|user|          timestamp|     prev_timestamp|gap_seconds|is_new_session|session_id|
+----+-------------------+-------------------+-----------+--------------+----------+
|   1|2026-03-29 10:00:00|               NULL|       NULL|             1|       1_1|
|   1|2026-03-29 10:15:00|2026-03-29 10:00:00|        900|             0|       1_1|
|   1|2026-03-29 10:50:00|2026-03-29 10:15:00|       2100|             1|       1_2|
|   1|2026-03-29 11:00:00|2026-03-29 10:50:00|        600|             0|       1_2|
|   2|2026-03-29 10:00:00|               NULL|       NULL|             1|       2_1|
|   2|2026-03-29 10:40:00|2026-03-29 10:00:00|       2400|             1|       2_2|
+----+-------------------+-------------------+-----------+--------------+----------+



8. Pivot Challenge

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Pivot").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
	('Jan','A', 10),
	('Jan', 'B', 15),
	('Feb','A', 12)
]

columns = ['month', 'product', 'sales']

sales = spark.createDataFrame(data=data, schema=columns)
sales.groupBy('month').pivot('product').sum('sales').show()



+-----+---+----+
|month|  A|   B|
+-----+---+----+
|  Feb| 12|NULL|
|  Jan| 10|  15|
+-----+---+----+



9. Explode Nested Data

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Explode").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [
  ("Alice", ["Math", "Science"]),
  ("Bob", ["History"])
]

columns = ['name', 'subject']

students = spark.createDataFrame(data=data, schema=columns)

students.withColumn('subject', func.explode(students.subject)).show()

+-----+-------+
| name|subject|
+-----+-------+
|Alice|   Math|
|Alice|Science|
|  Bob|History|
+-----+-------+



10. JSON Parsing

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Top N").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel('WARN')


data = [("""
		 {"customer":{"id":1,"city":"NY"}}
""",)]

customers = spark.createDataFrame(data=data, schema=["json_str"])

customer_schema = StructType([
    StructField("customer", StructType([
        StructField("id", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True)
])


parsed_customers = customers.withColumn("parsed_data", func.from_json(func.col("json_str"), customer_schema))

flattened_customers = parsed_customers.select(
    func.col("parsed_data.customer.id").alias("customer_id"),
    func.col("parsed_data.customer.city").alias("customer_city")
)

flattened_customers.show()






+-----------+-------------+
|customer_id|customer_city|
+-----------+-------------+
|          1|           NY|
+-----------+-------------+

